# Week 67, Day 1: multiprocessing

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w067d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def chunks_needed(n_items, chunksize):
    return -(-n_items // chunksize)          # ceiling division

def total_time(n_items, workers, chunksize, per_item=1, per_chunk=20):
    n_chunks = chunks_needed(n_items, chunksize)
    cost_per_chunk = per_chunk + chunksize * per_item     # overhead + work
    rounds = -(-n_chunks // workers)
    return rounds * cost_per_chunk

for size in (1, 10, 100, 1000):
    print(size, total_time(1000, workers=4, chunksize=size))

In [ ]:
import os

def crunch(n):
    return sum(i * i for i in range(n))

def simulated_map(func, items, workers):
    "What Pool.map promises: each item handled by some worker, results in input order."
    assignments = {i: f"worker {i % workers}" for i in range(len(items))}
    results = [func(x) for x in items]          # in reality these run in parallel processes
    return results, assignments

results, who = simulated_map(crunch, [10, 100, 1000], workers=2)
print(results, who)
print("this machine has", os.cpu_count() or "an unknown number of", "CPUs")

In [ ]:
import pickle

def sendable(obj):
    try:
        pickle.dumps(obj)
        return True
    except Exception:
        return False

print(sendable(len), sendable(lambda x: x), sendable((1, [2, "three"])))
print(sendable(x for x in range(3)))

data = list(range(100_000))
print(len(pickle.dumps(data)), "bytes to send this list to a worker, every time")
print(len(pickle.dumps(len)), "bytes for a function: just its module and name")
print(pickle.dumps(len)[-20:])

In [ ]:
def default_chunksize(n_items, workers):
    chunksize, extra = divmod(n_items, workers * 4)
    if extra:
        chunksize += 1
    return chunksize

for n in (10, 100, 1000, 1_000_000):
    print(n, "items on 4 workers ->", default_chunksize(n, 4), "per chunk")

def make_chunks(items, size):
    return [items[i:i + size] for i in range(0, len(items), size)]

print(make_chunks(list("abcdefg"), 3))

In [ ]:
import pickle
from collections import deque

class End:
    "One end of a simulated pipe: messages are pickled, as in a real one."
    def __init__(self):
        self.inbox, self.other, self.closed = deque(), None, False
    def send(self, obj):
        self.other.inbox.append(pickle.dumps(obj))
    def recv(self):
        if not self.inbox:
            raise EOFError("nothing to receive")
        return pickle.loads(self.inbox.popleft())
    def poll(self):
        return bool(self.inbox)

def pipe():
    a, b = End(), End()
    a.other, b.other = b, a
    return a, b

parent, child = pipe()
parent.send({"task": "resize", "size": (800, 600)})
job = child.recv()
child.send(f"resized to {job['size'][0]}x{job['size'][1]}")
print(parent.poll(), parent.recv(), parent.poll())

In [ ]:
def top_level_runs(guarded, children):
    "How many times does a script's top-level code run, including in spawned children?"
    runs = 1                                   # the parent
    runs += children                           # each spawned child imports the script
    starts_processes_in_children = not guarded
    return runs, ("children would start more processes" if starts_processes_in_children else "safe")

print(top_level_runs(guarded=True, children=4))
print(top_level_runs(guarded=False, children=4))

defaults = {"Windows": "spawn", "macOS": "spawn", "Linux (Python 3.14+)": "forkserver", "Linux (before 3.14)": "fork"}
for system, method in defaults.items():
    print(f"{system:22} {method}")

> **Key idea.** multiprocessing runs work in separate processes, each with its own interpreter and GIL: Process with start and join, or Pool with map (input order), imap, imap_unordered (finishing order), starmap and apply_async; ProcessPoolExecutor offers the same in the futures style. Everything sent between processes is pickled: functions by importable name (no lambdas or nested functions), arguments and results by copy, so keep them small. Work is sent in chunks: small chunks balance load, large ones cut overhead; Pool.map defaults to about four chunks per worker, while imap and executors default to one item. Pass messages with Queue and Pipe; share memory with Value, Array or shared_memory under a lock only when copying costs too much; Managers are flexible but slow. Start methods are fork, spawn (Windows and macOS) and forkserver (Linux from 3.14); spawned children import the main module, so guard process creation with if __name__ == "__main__":.

### Exercise: What can be sent?

Write `sendable(obj)`: True if `obj` can be pickled. Then `why_not(obj)`: `None` if it can be sent, otherwise a short reason chosen by type: `"lambdas and nested functions aren't importable"` for functions whose `__qualname__` contains `"<lambda>"` or `"<locals>"`, `"generators can't be pickled"` for generators, `"locks can't be pickled"` for objects whose type name ends in `"lock"` (any case), and `"can't be pickled"` otherwise. Finally `payload_size(obj)`: the number of bytes `pickle.dumps` produces.

In [ ]:
import pickle
import types

def sendable(obj):
    ___

def why_not(obj):
    ___

def payload_size(obj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pickling():\n    \"pickle rules\"\n    import threading\n    def nested():\n        return 1\n    def top_level_style(x):\n        return x\n    assert sendable([1, {\"a\": (2, 3)}]) and sendable(len) and not sendable(lambda: 0)\n    assert why_not(lambda: 0) == \"lambdas and nested functions aren't importable\"\n    assert why_not(nested) == \"lambdas and nested functions aren't importable\"\n    assert why_not(x for x in []) == \"generators can't be pickled\" and why_not(threading.Lock()) == \"locks can't be pickled\"\n    assert why_not(42) is None and payload_size(list(range(1000))) > payload_size(list(range(10)))")

### Exercise: Pool's default chunksize

Write `default_chunksize(n_items, workers)` using CPython's rule for `Pool.map` (divide by `workers * 4`, rounding up; and at least 1 when there are no items, so `map` still works), `make_chunks(items, size)`: the list of consecutive chunks of that size (the last may be shorter), and `messages(n_items, workers, chunksize=None)`: how many chunks (messages) `map` would send, using the default when `chunksize` is `None`.

In [ ]:
def default_chunksize(n_items, workers):
    ___

def make_chunks(items, size):
    ___

def messages(n_items, workers, chunksize=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_chunksize():\n    \"chunking rules\"\n    assert default_chunksize(100, 4) == 7 and default_chunksize(96, 4) == 6 and default_chunksize(1, 8) == 1\n    assert default_chunksize(0, 4) == 1\n    assert make_chunks(list(range(7)), 3) == [[0, 1, 2], [3, 4, 5], [6]] and make_chunks([], 2) == []\n    assert messages(100, 4) == 15 and messages(100, 4, chunksize=1) == 100 and messages(0, 4) == 0")

### Exercise: Simulating Pool.map

Simulate `Pool.map` in time. `pool_map(func, items, workers, chunksize, cost, overhead)`: split the items into chunks; each chunk goes, in order, to the worker that's free first (lowest-numbered on ties) and takes `overhead + sum(cost(x) for x in chunk)` time units. Return `(results, finish)`: the results of `func` in **input order**, and when the last chunk finishes. Then `best_chunksize(items, workers, cost, overhead, candidates)`: the candidate chunksize with the earliest finish (the smallest such size on ties).

In [ ]:
def pool_map(func, items, workers, chunksize, cost, overhead):
    ___

def best_chunksize(items, workers, cost, overhead, candidates):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_poolsim():\n    \"chunk scheduling\"\n    items = list(range(8))\n    results, finish = pool_map(lambda x: x * x, items, 2, 2, cost=lambda x: 1, overhead=3)\n    assert results == [x * x for x in items] and finish == 10\n    assert pool_map(str, items, 2, 1, lambda x: 1, 3)[1] == 16 and pool_map(str, items, 2, 8, lambda x: 1, 3)[1] == 11\n    assert best_chunksize(items, 2, lambda x: 1, 3, [1, 2, 4, 8]) == 4\n    uneven = [10, 1, 1, 1, 1, 1, 1, 1]\n    assert best_chunksize(uneven, 2, lambda x: x, 1, [1, 2, 4, 8]) == 2\n    assert pool_map(abs, [], 3, 2, lambda x: 1, 5) == ([], 0)")

### Exercise: A pipe between processes

Simulate `multiprocessing.Pipe()`. Write `pipe()` returning two connected `Connection` objects: `send(obj)` pickles the object and puts the bytes in the **other** end's inbox; `recv()` returns the next message, unpickled (a **copy**), raising `EOFError` if nothing is waiting and the other end is closed, and `BlockingIOError` if nothing is waiting but the other end is still open (a real pipe would wait); `poll()` says whether a message is waiting; `close()` closes this end, after which `send` raises `OSError("handle is closed")`. `sent` counts bytes this end has sent.

In [ ]:
import pickle
from collections import deque

class Connection:
    def __init__(self):
        ___

def pipe():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pipes():\n    \"message passing\"\n    a, b = pipe()\n    job = {\"items\": [1, 2, 3]}\n    a.send(job)\n    a.send(\"second\")\n    assert b.poll() and not a.poll()\n    received = b.recv()\n    assert received == job and received is not job and b.recv() == \"second\" and not b.poll()\n    b.send(sum(received[\"items\"]))\n    assert a.recv() == 6 and a.sent > 0\n    try:\n        a.recv()\n        assert False\n    except BlockingIOError:\n        pass\n    b.close()\n    try:\n        a.recv()\n        assert False\n    except EOFError:\n        pass\n    try:\n        b.send(1)\n        assert False\n    except OSError as e:\n        assert str(e) == \"handle is closed\"\n    try:\n        a.send(lambda: 1)\n        assert False\n    except Exception:\n        pass")

### Exercise: Why the main guard matters

Model what happens when a script is started with the **spawn** method. Write `spawned_runs(guarded, workers, depth=3)`: the number of times the script's top-level code runs. The parent runs it once and starts `workers` children; each child imports the script, running its top-level code once; if the process-starting code is **not** guarded, each child also starts `workers` children of its own, and so on, stopping (as Python does, with an error) after `depth` levels of children. Then `start_method(system, version)`: the default method, `"spawn"` for `"Windows"` and `"macOS"`, and for `"Linux"` `"forkserver"` from version `(3, 14)` on and `"fork"` before.

In [ ]:
def spawned_runs(guarded, workers, depth=3):
    ___

def start_method(system, version):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_guard():\n    \"spawn and the __main__ guard\"\n    assert spawned_runs(True, 4) == 5 and spawned_runs(True, 1) == 2\n    assert spawned_runs(False, 2, depth=3) == 1 + 2 + 4 + 8 and spawned_runs(False, 4, depth=1) == 5\n    assert start_method(\"Windows\", (3, 12)) == \"spawn\" and start_method(\"macOS\", (3, 14)) == \"spawn\"\n    assert start_method(\"Linux\", (3, 13)) == \"fork\" and start_method(\"Linux\", (3, 14)) == \"forkserver\"")

### Bug hunt: A lambda that can't travel

The work is passed as a lambda, but a process pool sends functions by **name** (that's how pickle stores them), and a lambda has no name a worker could import. This simulated pool checks that, as a real one would, and the program crashes with `TypeError`. Define the function at the top level of the module with `def` so it can be sent, and this prints `[1, 4, 9]`.

In [ ]:
def simulated_pool_map(func, items):
    if "<lambda>" in func.__qualname__ or "<locals>" in func.__qualname__:
        raise TypeError(f"can't send {func.__qualname__}: workers can't import it by name")
    return [func(x) for x in items]

print(simulated_pool_map(lambda x: x * x, [1, 2, 3]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"importable functions\"\n    expect_output(\"[1, 4, 9]\")")

### Bug hunt: A worker that changed a copy

The worker appends its result to the list it was given, expecting the parent to see it, but processes receive **copies**, so the parent's list is still empty and this prints `[]`. Return the result from the worker and collect it in the parent, so this prints `[30]`.

In [ ]:
import copy

def worker(results, x):
    results.append(x * 10)

def run_in_process(func, *args):
    return func(*copy.deepcopy(args))            # a process gets copies of its arguments

results = []
run_in_process(worker, results, 3)
print(results)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"return results\"\n    expect_output(\"[30]\")")

### Bug hunt: A remainder left behind

The chunksize is computed with plain division, rounding down, so a few items at the end never get a chunk of their own and only `96` of the 100 items are sent. Round up, as `Pool.map` does, so all of them go and this prints `100`.

In [ ]:
items = list(range(100))
workers = 4
size = len(items) // (workers * 4)
chunks = [items[i * size:(i + 1) * size] for i in range(workers * 4)]
print(sum(len(c) for c in chunks))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"round up\"\n    expect_output(\"100\")")

### Puzzle: Using a pool safely

Arrange the lines of a script that computes squares in a process pool, with the guard that spawned children need (as you'd write it on your own computer). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: a worker process can change the parent's list that it was passed
q = ___   # Pool.map's default chunksize for 100 items on 5 workers
r = ___   # the start method on Windows
s = ___   # the order imap_unordered yields results in: "input" or "finishing"
t = ___   # True or False: a lambda can be sent to a process pool

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is False and q == 5 and r == \"spawn\", \"p, q and r\"\n    assert s == \"finishing\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: map and reduce

**MapReduce** splits a big job into a **map** step run in parallel on chunks of the data and a **reduce** step that combines the partial results. Count word frequencies across many documents this way, with the parallel part simulated:

- `map_chunk(docs)`: a `collections.Counter` of lower-cased words (split on whitespace, with `.,!?;:` stripped from each word's ends, empty words skipped) in a list of documents. It must be a **top-level** function, so it could be sent to a real pool.
- `run_job(docs, workers, chunksize)`: split the documents into chunks, "send" each chunk to a worker by pickling `("map_chunk", chunk)` (a function travels as its name, which is what pickle does with functions) and unpickling it, look the function up by name in the dict `FUNCTIONS = {"map_chunk": map_chunk}` (as a worker importing it would), apply it, pickle the partial result back, and finally merge all the partial counters in chunk order. Return `(total_counter, bytes_sent)`, where `bytes_sent` is the total size of everything pickled in both directions.
- `top_words(counter, k)`: the `k` most common `(word, count)` pairs, ties alphabetical.

The result must equal a single-process count, whatever the chunksize.

In [ ]:
import pickle
from collections import Counter

def map_chunk(docs):
    ___

def run_job(docs, workers, chunksize):
    ___

def top_words(counter, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_mapreduce():\n    \"a parallel word count\"\n    docs = [\"The cat sat.\", \"The dog sat!\", \"A cat, a dog; the end?\"] * 20\n    single = map_chunk(docs)\n    assert single[\"the\"] == 60 and single[\"cat\"] == 40 and single[\"end\"] == 20 and \"\" not in single\n    for size in (1, 7, 60):\n        total, sent = run_job(docs, workers=3, chunksize=size)\n        assert total == single and sent > 0\n    _, small_chunks = run_job(docs, 3, 1)\n    _, big_chunks = run_job(docs, 3, 60)\n    assert small_chunks > big_chunks\n    assert top_words(single, 3) == [(\"the\", 60), (\"a\", 40), (\"cat\", 40)]\n    assert run_job([], 2, 5) == (Counter(), 0)")

## Warm-up

Sending items to worker processes one at a time pays the overhead a thousand times; sending them in a few huge chunks leaves workers idle. The best chunk size is in between, which is why `Pool.map` has a `chunksize` argument.

## Process and Pool

`multiprocessing` mirrors `threading`, but each worker is a separate **process**:

```
import multiprocessing as mp

def crunch(n):
    return sum(i * i for i in range(n))

if __name__ == "__main__":                       # required: see "Start methods"
    p = mp.Process(target=crunch, args=(10_000_000,))
    p.start()
    p.join()                                      # the return value is lost, as with threads
    print("exit code", p.exitcode)

    with mp.Pool(processes=4) as pool:            # defaults to os.cpu_count()
        print(pool.map(crunch, [10**6] * 8))      # results in input order
        print(pool.apply(crunch, (1000,)))        # one call, waiting for it
        for r in pool.imap_unordered(crunch, [10**5] * 4):
            print(r)                              # as they finish
```

- **`Pool.map(func, items, chunksize=None)`** splits `items` into chunks, sends them to workers and returns a list of results in input order. `imap` returns them lazily in order; `imap_unordered` as they finish; `starmap` unpacks argument tuples; `apply_async` and `map_async` return result objects with `.get()`.
- `concurrent.futures.ProcessPoolExecutor` (last Friday) offers the same in the futures style, and is usually the simpler choice for new code.
- A worker that crashes outright (rather than raising) breaks a `ProcessPoolExecutor` (`BrokenProcessPool`); exceptions raised normally are pickled and re-raised in the parent.

## What travels between processes

Processes don't share memory, so everything that crosses between them is **pickled** in one process and unpickled in the other:

- the **function** to call (pickled by **reference**: its module and name, so the worker must be able to import it; that's why lambdas, nested functions and functions defined in the interactive prompt fail);
- the **arguments** and the **results** (pickled by value: copies, so changes a worker makes to an argument never reach the parent);
- **exceptions** raised in workers.

Consequences: keep arguments and results **small** (pickling a big list for every task costs more than the work); things like open files, sockets, locks and generators can't be sent; and global variables changed in the parent after the pool starts aren't seen by workers. Send what each task needs, return what it produces, and nothing else.

## Chunks

Each message between processes has a fixed cost (pickling, a system call, waking the worker), so `Pool.map` sends items in **chunks**. The trade-off:

- **Small chunks**: good **balance** (a worker that finishes early takes more), but the overhead is paid many times.
- **Large chunks**: little overhead, but poor balance: if items take different times, one worker may be stuck with a slow chunk while the others sit idle at the end.

CPython's `Pool.map` picks a default by dividing the items into about **four chunks per worker**:

```
chunksize, extra = divmod(len(items), len(workers) * 4)
if extra:
    chunksize += 1
```

`imap` and `imap_unordered` default to a chunksize of 1, and `ProcessPoolExecutor.map` defaults to 1 too, so pass `chunksize` yourself for many small tasks.

## Talking between processes

Prefer passing messages to sharing memory:

- **`multiprocessing.Queue()`**: like `queue.Queue` but between processes (items are pickled). The producer-consumer pattern with sentinels works the same way.
- **`multiprocessing.Pipe()`**: returns two **connection** objects, one for each end; `conn.send(obj)` and `conn.recv()` (pickled messages), `conn.poll(timeout)` checks for waiting data, and `recv()` on a closed pipe raises `EOFError`. Faster than a queue for exactly two parties.
- **Shared memory** when copying is too costly: `Value("i", 0)` and `Array("d", 100)` hold C values in memory all processes can see (with a lock: `with v.get_lock(): v.value += 1`, since races happen across processes too), and `multiprocessing.shared_memory.SharedMemory` gives a raw shared block (NumPy can use it directly).
- **`Manager()`**: a server process holding ordinary Python objects (dicts, lists) that other processes use through proxies. Flexible but slow: every access is a message.

## Start methods

How a new process begins matters more than it sounds:

- **fork** (the classic Unix way): the child is a copy of the parent at that moment, including all its memory. Fast, and children inherit already-loaded data. But copying a process that has threads is unsafe (a lock held by another thread is copied as locked for ever), so it can deadlock in programs that use threads or some libraries.
- **spawn** (the default on Windows and macOS): the child starts a **fresh** Python interpreter, imports your main module, and receives the function and arguments by pickling. Safer, slower to start.
- **forkserver**: a clean server process is started once, and new workers are forked from it. Python 3.14 made it the default on Linux, replacing fork.

With **spawn** and **forkserver**, the child **imports your script**. Code at the top level of the script runs again in every child, so creating processes there would create processes for ever: that's why the code that starts processes must be inside **`if __name__ == "__main__":`**. Choose explicitly with `multiprocessing.set_start_method("spawn")` once at start-up, or `multiprocessing.get_context("spawn")`.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, run the "Process and Pool" example and compare `pool.map(crunch, ...)` with a plain loop for eight tasks of ten million: with four or more cores, the pool should be several times faster. Tomorrow: `asyncio`, a different kind of concurrency, built on coroutines.